[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_41_Model_Merging.ipynb)

# Lesson 41 — Model Merging: SLERP, TIES & DARE
### Track 3 · Self-hosted & Fine-tuning — Lesson 5 of 5 ✅ FINAL

**Learning objectives:**
- Understand *why* merging beats ensembling for zero-cost capability combination
- Implement SLERP from first principles — geometry in weight space
- Understand TIES (trim → elect → merge) and DARE (drop → rescale) algorithms
- Run a real merge with **mergekit** using YAML config
- Evaluate merged vs individual models; know when merging fails

**Prerequisites:** L37 (vLLM), L38 (QLoRA), L39 (DPO/ORPO), L40 (Distillation)  
**Compute needed:** T4 (free Colab) — math demos need no GPU; mergekit needs ~8 GB VRAM

---
> 📍 **Track 3 Progress:** `vLLM ✅` → `QLoRA ✅` → `DPO/ORPO ✅` → `Distillation ✅` → **Model Merging ← you are here**


In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
# Run once; restart runtime if prompted after install
!pip install anthropic mergekit transformers peft torch numpy matplotlib pandas pyyaml -q

import os, json, re, math, copy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional

# API key ── Colab Secrets (key icon in left sidebar)
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('✅ API key loaded from Colab Secrets')
except Exception:
    print("⚠️  Set ANTHROPIC_API_KEY manually: os.environ['ANTHROPIC_API_KEY'] = 'sk-...'")

import anthropic
client = anthropic.Anthropic()

HAIKU  = 'claude-haiku-4-5'
SONNET = 'claude-sonnet-4-6'
print(f'Models: HAIKU={HAIKU}  SONNET={SONNET}')


## § 1 — Why Merge Models?

After Track 3 you have several fine-tuned adapters:
- **L38:** QLoRA SQL adapter
- **L39:** DPO-tuned SQL adapter (better preference alignment)
- **L40:** Student model distilled from teacher

**Problem:** These represent *different* optimizations on the *same* base model. You want **all** improvements at once.

| Strategy | How | Cost | Tradeoffs |
|---|---|---|---|
| **Ensemble** | Run all models, aggregate outputs | N× inference | Expensive, not a single model |
| **Multi-task SFT** | Retrain on combined data | GPU hours + data | Risk of task interference |
| **Distillation** | New student trained on all teachers | GPU hours | Best quality but needs training |
| **Model Merge** ⭐ | Interpolate weights in parameter space | **Free** (no training) | Slight quality trade-off, instant |

**Key insight:** Fine-tuned models live in the same *neighborhood* of weight space as their shared base model. The fine-tuning delta (ΔW = W_fine - W_base) is small and sparse. We can *add* multiple deltas without full retraining:

```
W_merged = W_base + α·ΔW_A + β·ΔW_B
```

This is the core idea behind SLERP, TIES, and DARE.


In [ ]:
# ── §1: Visualize weight-space neighborhoods ─────────────────────────────
np.random.seed(42)

W_base = np.array([0.0, 0.0])
W_A = W_base + np.array([0.8, 0.3])   # QLoRA SQL
W_B = W_base + np.array([0.5, 0.9])   # DPO SQL

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
models = {'Base': W_base, 'QLoRA (A)': W_A, 'DPO (B)': W_B}
colors = {'Base': 'blue', 'QLoRA (A)': 'green', 'DPO (B)': 'orange'}
for name, pos in models.items():
    ax.scatter(*pos, s=200, color=colors[name], zorder=5, label=name)
    ax.annotate(name, pos, textcoords='offset points', xytext=(8,5), fontsize=11)
ax.annotate('', xy=W_A, xytext=W_base,
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.annotate('', xy=W_B, xytext=W_base,
            arrowprops=dict(arrowstyle='->', color='orange', lw=2))
ax.text(0.3, 0.0, 'ΔW_A', color='green', fontsize=10)
ax.text(-0.2, 0.5, 'ΔW_B', color='orange', fontsize=10)
W_linear = 0.5 * W_A + 0.5 * W_B
ax.scatter(*W_linear, s=200, color='red', zorder=5, marker='*', label='Linear merge (t=0.5)')
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.set_title('Models live near base in weight space', fontsize=12)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.set_xlabel('Weight dim 1'); ax.set_ylabel('Weight dim 2')

ax2 = axes[1]
layers = [f'L{i}' for i in range(1, 13)]
delta_A = np.abs(np.random.normal(0.02, 0.008, 12))
delta_B = np.abs(np.random.normal(0.025, 0.009, 12))
delta_rnd = np.abs(np.random.normal(0.5, 0.1, 12))
x = np.arange(12); w = 0.25
ax2.bar(x-w, delta_A,   w, label='QLoRA ΔW (small)', color='green',  alpha=0.8)
ax2.bar(x,   delta_B,   w, label='DPO ΔW (small)',   color='orange', alpha=0.8)
ax2.bar(x+w, delta_rnd, w, label='Random (huge)',     color='red',    alpha=0.8)
ax2.set_xticks(x); ax2.set_xticklabels(layers, fontsize=8)
ax2.set_ylabel('Mean |ΔW| per layer')
ax2.set_title('Fine-tuned deltas are tiny vs random\n→ safe to combine', fontsize=12)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout(); plt.savefig('weight_space.png', dpi=110, bbox_inches='tight')
plt.show()
print('Key insight: ΔW_QLoRA ≈ 0.02 vs random ≈ 0.5 — fine-tuning barely moves weights!')
# 💡 EXPERIMENT: What happens if you try to merge two completely different base models?


## § 2 — SLERP: Spherical Linear Interpolation

**Linear interpolation** (LERP) mixes two points on a *straight line*:
```
LERP(A, B, t) = (1-t)·A + t·B
```

**Problem:** straight-line interpolation passes *through* the interior of the hypersphere, shrinking the magnitude at the midpoint.

**SLERP** interpolates along the *great circle* arc:
```
SLERP(A, B, t) = A·sin((1-t)·Ω)/sin(Ω)  +  B·sin(t·Ω)/sin(Ω)
where Ω = arccos(A·B / (|A|·|B|))
```

- `t=0` → pure model A; `t=1` → pure model B; `t=0.5` → equal blend
- When Ω → 0 (A ≈ B), SLERP degenerates to LERP safely
- SLERP produces smoother, more isotropic interpolation paths than LERP


In [ ]:
# ── §2: SLERP from scratch ───────────────────────────────────────────────

def slerp(v0: np.ndarray, v1: np.ndarray, t: float, eps: float = 1e-8) -> np.ndarray:
    """Spherical linear interpolation between two weight tensors."""
    n0 = v0 / (np.linalg.norm(v0) + eps)
    n1 = v1 / (np.linalg.norm(v1) + eps)
    dot = np.clip(np.dot(n0.flatten(), n1.flatten()), -1.0, 1.0)
    omega = np.arccos(dot)
    if abs(omega) < eps:
        return (1.0 - t) * v0 + t * v1
    sin_omega = np.sin(omega)
    return np.sin((1.0 - t) * omega) / sin_omega * v0 + np.sin(t * omega) / sin_omega * v1


def lerp(v0: np.ndarray, v1: np.ndarray, t: float) -> np.ndarray:
    return (1.0 - t) * v0 + t * v1


A = np.array([1.0, 0.0])
B = np.array([0.0, 1.0])
ts = np.linspace(0, 1, 50)
lerp_path  = np.array([lerp(A, B, t)  for t in ts])
slerp_path = np.array([slerp(A, B, t) for t in ts])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
theta = np.linspace(0, np.pi/2, 100)
ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.4, label='Unit circle')
ax.plot(lerp_path[:,0],  lerp_path[:,1],  'b-', lw=2.5, label='LERP (straight)')
ax.plot(slerp_path[:,0], slerp_path[:,1], 'r-', lw=2.5, label='SLERP (arc)')
ax.scatter(*A, s=200, color='green', zorder=5); ax.annotate('Model A', A, xytext=(-40,10), textcoords='offset points')
ax.scatter(*B, s=200, color='orange', zorder=5); ax.annotate('Model B', B, xytext=(5,5), textcoords='offset points')
ax.scatter(*lerp(A,B,0.5),  s=200, color='blue',  zorder=5, marker='*')
ax.scatter(*slerp(A,B,0.5), s=200, color='red',   zorder=5, marker='*')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('LERP vs SLERP path (2D)', fontsize=12)
ax.legend(fontsize=10); ax.set_xlim(-0.1, 1.2); ax.set_ylim(-0.1, 1.2)

ax2 = axes[1]
lerp_norm  = [np.linalg.norm(lerp(A, B, t))  for t in ts]
slerp_norm = [np.linalg.norm(slerp(A, B, t)) for t in ts]
ax2.plot(ts, lerp_norm,  'b-', lw=2.5, label='LERP magnitude')
ax2.plot(ts, slerp_norm, 'r-', lw=2.5, label='SLERP magnitude')
ax2.axhline(1.0, color='k', linestyle='--', alpha=0.4, label='Unit norm')
ax2.set_xlabel('t (0=A, 1=B)'); ax2.set_ylabel('||merged vector||')
ax2.set_title('LERP shrinks magnitude at midpoint!\nSLERP preserves it', fontsize=12)
ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig('slerp_vs_lerp.png', dpi=110, bbox_inches='tight'); plt.show()

mid_lerp  = lerp(A, B, 0.5)
mid_slerp = slerp(A, B, 0.5)
print(f'LERP  midpoint: {mid_lerp}  norm={np.linalg.norm(mid_lerp):.4f}  <- shrinks!')
print(f'SLERP midpoint: {mid_slerp}  norm={np.linalg.norm(mid_slerp):.4f}  <- preserved!')
# 💡 EXPERIMENT: Try t=0.3 (bias toward A) vs t=0.7 (bias toward B)


## § 3 — TIES: Trim → Elect → Merge

**Paper:** "TIES-Merging: Resolving Interference When Merging Models" (Yadav et al., 2023)

**Problem:** When you naively add multiple delta vectors, opposing-sign updates at the same parameter *cancel out* or *amplify destructively*.

**TIES algorithm — 3 steps:**

```
Step 1: TRIM
  For each delta ΔW = W_fine - W_base:
  Keep only the top-k% largest absolute values → zero out the rest
  (density=0.3 → keep top 30% of parameters)

Step 2: ELECT
  For each parameter position, count positive vs negative signs across models.
  Elect majority sign: if more models are positive → elected_sign = +1

Step 3: DISJOINT MERGE
  Only average deltas that AGREE with elected sign.
  W_merged = W_base + mean(agreeing_deltas)
```

**Why TRIM?** Fine-tuning moves only a small fraction of params meaningfully. The rest is noise that interferes with other models.

**Why ELECT?** ΔW_A[i]=+0.1 and ΔW_B[i]=-0.1 would cancel to 0. Electing the majority sign preserves both models' intent.


In [ ]:
# ── §3: TIES algorithm from scratch ──────────────────────────────────────

def ties_merge(
    base: np.ndarray,
    fine_tuned_models: List[np.ndarray],
    density: float = 0.3,
    scaling: float = 1.0,
) -> np.ndarray:
    """
    TIES-Merging (Yadav et al. 2023).
    density: fraction of delta params to keep (lower = more aggressive trim)
    """
    deltas = [ft - base for ft in fine_tuned_models]
    n_params = base.size
    k = max(1, int(n_params * density))

    # ── STEP 1: TRIM ──────────────────────────────────────────────────────
    trimmed = []
    for delta in deltas:
        flat = delta.flatten()
        threshold = np.partition(np.abs(flat), -k)[-k]
        trimmed.append(np.where(np.abs(delta) >= threshold, delta, 0.0))

    # ── STEP 2: ELECT (majority sign per position) ────────────────────────
    sign_stack = np.sign(np.stack(trimmed, axis=0))  # (n_models, ...)
    pos_count  = (sign_stack > 0).sum(axis=0)
    neg_count  = (sign_stack < 0).sum(axis=0)
    elected    = np.where(pos_count >= neg_count, 1.0, -1.0)

    # ── STEP 3: DISJOINT MERGE ────────────────────────────────────────────
    agree_sum = np.zeros_like(base)
    agree_cnt = np.zeros_like(base)
    for td in trimmed:
        mask = (np.sign(td) == elected) & (td != 0)
        agree_sum += np.where(mask, td, 0.0)
        agree_cnt += mask.astype(float)
    merged_delta = np.where(agree_cnt > 0, agree_sum / (agree_cnt + 1e-8), 0.0)
    return base + scaling * merged_delta


# ── Demo ──────────────────────────────────────────────────────────────────
np.random.seed(0)
D = 100
base  = np.random.randn(D) * 0.5
delta_qlora = np.zeros(D)
delta_qlora[:30] = np.random.randn(30) * 0.3
delta_dpo = np.zeros(D)
delta_dpo[20:50] = np.random.randn(30) * 0.25
delta_dpo[20:30] *= -1   # sign conflict in overlap region

model_A = base + delta_qlora
model_B = base + delta_dpo
naive_merged = 0.5 * model_A + 0.5 * model_B
ties_merged  = ties_merge(base, [model_A, model_B], density=0.3)
ideal        = base + delta_qlora + delta_dpo

def recovery_score(merged, ideal, base):
    id_delta = ideal - base
    mg_delta = merged - base
    if np.linalg.norm(id_delta) < 1e-8: return 1.0
    return 1.0 - np.linalg.norm(mg_delta - id_delta) / np.linalg.norm(id_delta)

print('=== TIES vs Naive Merge ===')
print(f'  Naive merge recovery: {recovery_score(naive_merged, ideal, base):.4f}')
print(f'  TIES  merge recovery: {recovery_score(ties_merged,  ideal, base):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(D)
axes[0].plot(x, delta_qlora, 'g-', alpha=0.7, label='ΔW QLoRA', lw=1.5)
axes[0].plot(x, delta_dpo,   color='orange', alpha=0.7, label='ΔW DPO',   lw=1.5)
axes[0].axvspan(20, 30, alpha=0.15, color='red', label='Sign conflict')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
axes[0].set_title('Delta vectors — sign conflict in overlap', fontsize=11)
axes[0].set_xlabel('Weight parameter index')

axes[1].plot(x, ideal-base,        'k-',  alpha=0.6, label='Ideal delta', lw=2)
axes[1].plot(x, naive_merged-base, 'b--', alpha=0.8, label='Naive merge', lw=1.5)
axes[1].plot(x, ties_merged-base,  'r-',  alpha=0.8, label='TIES merge',  lw=1.5)
axes[1].axvspan(20, 30, alpha=0.15, color='red')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
axes[1].set_title('Recovery of combined delta', fontsize=11)
axes[1].set_xlabel('Weight parameter index')

plt.tight_layout(); plt.savefig('ties_demo.png', dpi=110, bbox_inches='tight'); plt.show()
# 💡 EXPERIMENT: Change density=0.1 (aggressive) vs density=0.8 (keep most)


## § 4 — DARE: Drop And REscale

**Paper:** "Language Models are Super Mario" (Yu et al., 2023)

**Observation:** Most delta weights are small and *redundant*. Randomly dropping them (with rescaling) reduces interference without hurting performance.

**DARE algorithm:**
```
For each parameter in ΔW:
  1. DROP with probability p  →  set to 0
  2. RESCALE remaining:  ΔW_dare = ΔW / (1 - p)
     (rescaling preserves expected value of the delta)
  3. Use ΔW_dare in downstream TIES merge
```

**DARE-TIES (combined) — the production default:**
```
ΔW  →  DARE (random drop p=0.9 + rescale)  →  TIES (trim + elect + merge)
```

| Param | Typical | Effect |
|---|---|---|
| `p` (drop rate) | 0.9 | 90% dropped, 10x rescale |
| `density` (TIES) | 0.2 | Top 20% of remaining delta kept |
| `t` (SLERP) | 0.5 | Equal blend |


In [ ]:
# ── §4: DARE + DARE-TIES from scratch ───────────────────────────────────

def dare_delta(delta: np.ndarray, p: float = 0.9, seed: int = 42) -> np.ndarray:
    """DARE: randomly drop p fraction of delta params, then rescale."""
    rng = np.random.default_rng(seed)
    mask    = rng.random(delta.shape) >= p  # True = keep
    rescale = 1.0 / (1.0 - p + 1e-8)
    return delta * mask * rescale


def dare_ties_merge(
    base: np.ndarray,
    fine_tuned_models: List[np.ndarray],
    dare_p: float = 0.9,
    ties_density: float = 0.5,
    scaling: float = 1.0,
) -> np.ndarray:
    """Full DARE-TIES pipeline."""
    dare_deltas = [dare_delta(ft - base, p=dare_p, seed=i*100)
                   for i, ft in enumerate(fine_tuned_models)]
    dare_models = [base + d for d in dare_deltas]
    return ties_merge(base, dare_models, density=ties_density, scaling=scaling)


# ── Compare all merge strategies on 3 models ──────────────────────────────
np.random.seed(42)
D = 200
base = np.random.randn(D) * 0.5
deltas = []
for i in range(3):
    d = np.random.randn(D) * 0.005  # noise baseline
    start = i * 30
    d[start:start+60] = np.random.randn(60) * 0.2  # meaningful changes
    deltas.append(d)
models = [base + d for d in deltas]
ideal  = base + sum(deltas)

results = {
    'Naive LERP (equal 1/3)':    sum(m/3 for m in models),
    'SLERP (A+B blend)':         base + slerp(deltas[0], deltas[1], 0.5) + deltas[2]*0.5,
    'TIES (density=0.3)':        ties_merge(base, models, density=0.3),
    'DARE-TIES (p=0.9, d=0.3)':  dare_ties_merge(base, models, dare_p=0.9, ties_density=0.3),
}

print('=== Multi-model Merge Comparison ===')
print(f'{"Strategy":<35} {"Recovery":>10} {"Delta Sparsity":>16}')
print('-' * 65)
for name, merged in results.items():
    rec      = recovery_score(merged, ideal, base)
    sparsity = (np.abs(merged - base) < 1e-6).mean()
    print(f'  {name:<33} {rec:>10.4f} {sparsity:>14.1%}')

fig, ax = plt.subplots(figsize=(9, 4))
methods = list(results.keys())
scores  = [recovery_score(r, ideal, base) for r in results.values()]
colors  = ['#4C72B0', '#55A868', '#C44E52', '#8172B3']
bars    = ax.bar(range(len(methods)), scores, color=colors, alpha=0.85, edgecolor='white', lw=1.5)
ax.set_xticks(range(len(methods))); ax.set_xticklabels(methods, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Recovery score (higher = better)')
ax.set_title('Merge strategy comparison — 3 fine-tuned models', fontsize=12)
ax.set_ylim(0, 1.05)
for bar, s in zip(bars, scores):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{s:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig('merge_comparison.png', dpi=110, bbox_inches='tight'); plt.show()
# 💡 EXPERIMENT: Change dare_p from 0.5 to 0.95 — at what drop rate does DARE hurt?


## § 5 — Mergekit: Production-Grade Model Merging

[mergekit](https://github.com/arcee-ai/mergekit) is the de-facto tool for merging real LLMs. It handles:
- All major merge methods: `linear`, `slerp`, `ties`, `dare_ties`, `breadcrumbs`, `task_arithmetic`
- Large models (offload to CPU/disk when GPU VRAM insufficient)
- LoRA adapters and merged models
- HuggingFace Hub integration

**Configuration is pure YAML. Example — SLERP of two SQL models:**

```yaml
merge_method: slerp
base_model: Qwen/Qwen2.5-0.5B-Instruct
models:
  - model: ./model_qlora_merged     # your L38 output
    parameters:
      t: 0.4                        # 40% toward QLoRA
  - model: ./model_dpo_merged       # your L39 output
    parameters:
      t: 0.6                        # 60% toward DPO
dtype: bfloat16
out_path: ./my_slerp_model
```

**DARE-TIES config (use when merging 3+ models):**

```yaml
merge_method: dare_ties
base_model: Qwen/Qwen2.5-0.5B-Instruct
models:
  - model: ./qlora_merged
    parameters:
      density: 0.5
      weight: 1.0
  - model: ./dpo_merged
    parameters:
      density: 0.5
      weight: 1.0
  - model: ./distilled_student      # from L40!
    parameters:
      density: 0.4
      weight: 0.8
parameters:
  normalize: true
dtype: bfloat16
out_path: ./dare_ties_all_three
```

**Run command:**
```bash
mergekit-yaml merge_config.yml ./output/ --cuda --copy-tokenizer
# Add --low-cpu-memory for 7B+ models on T4
```


In [ ]:
# ── §5: Mergekit config generation + internals explanation ───────────────
import yaml, os

MERGE_DIR = '/content/merge_demo'
os.makedirs(MERGE_DIR, exist_ok=True)

# Write the SLERP config
slerp_config = {
    'merge_method': 'slerp',
    'base_model': 'Qwen/Qwen2.5-0.5B-Instruct',
    'models': [
        {'model': './qlora_sql_merged', 'parameters': {
            't': [{'filter': 'self_attn', 'value': 0.3},
                  {'filter': 'mlp',       'value': 0.7},
                  {'value': 0.5}]
        }}
    ],
    'dtype': 'bfloat16',
    'out_path': f'{MERGE_DIR}/slerp_output',
    'tokenizer_source': 'base'
}
with open(f'{MERGE_DIR}/slerp_config.yml', 'w') as f:
    yaml.dump(slerp_config, f, default_flow_style=False)

dare_ties_config = {
    'merge_method': 'dare_ties',
    'base_model': 'Qwen/Qwen2.5-0.5B-Instruct',
    'models': [
        {'model': './qlora_sql_merged',   'parameters': {'density': 0.5, 'weight': 1.0}},
        {'model': './dpo_sql_merged',     'parameters': {'density': 0.5, 'weight': 1.0}},
        {'model': './distilled_student',  'parameters': {'density': 0.4, 'weight': 0.8}},
    ],
    'parameters': {'normalize': True},
    'dtype': 'bfloat16',
    'out_path': f'{MERGE_DIR}/dare_ties_output',
    'tokenizer_source': 'base'
}
with open(f'{MERGE_DIR}/dare_ties_config.yml', 'w') as f:
    yaml.dump(dare_ties_config, f, default_flow_style=False)

print('Config files written:')
for fn in os.listdir(MERGE_DIR):
    print(f'  {MERGE_DIR}/{fn}')

print()
print('To run SLERP merge (replace with your real model paths):')
print(f'  !mergekit-yaml {MERGE_DIR}/slerp_config.yml ./merged_slerp/ --cuda --copy-tokenizer')
print()
print('To run DARE-TIES 3-way merge:')
print(f'  !mergekit-yaml {MERGE_DIR}/dare_ties_config.yml ./merged_dare_ties/ --cuda --copy-tokenizer --low-cpu-memory')

print()
print('''
Mergekit internals (what runs under the hood):
  1. Load base model + each fine-tuned model (with disk offloading for large models)
  2. For each weight tensor layer-by-layer:
     a. Compute delta: dW = W_fine - W_base
     b. Apply method (SLERP/TIES/DARE) per tensor
     c. Write output tensor immediately (streaming — low peak VRAM)
  3. Copy tokenizer + config.json from base model
  4. Save in safetensors format (HF-compatible, mmap-safe)

VRAM budget:
  0.5B BF16: ~3 GB  |  1.5B BF16: ~7 GB  |  7B BF16: ~16 GB (needs --low-cpu-memory on T4)
''')
# 💡 EXPERIMENT: Replace model paths above with your real L38/L39 outputs
# and run the mergekit-yaml command — your merged model is ready in <5 minutes!


## § 6 — Evaluating a Merged Model

After merging, the key question: **did we keep capabilities from both source models?**

Evaluation strategy (same as L38–L40 SQL harness):
1. **Heuristic SQL score** — structural checks (SELECT/FROM/WHERE, no syntax placeholders)
2. **LLM judge** — Haiku grades correctness, schema adherence, efficiency
3. **Comparison table** — merged vs QLoRA vs DPO vs base

**What a good merge looks like:**
- `Score(merged) ≥ 0.9 × max(Score(A), Score(B))` → acceptable merge (no major regression)
- `Score(merged) ≥ mean(Score(A), Score(B)) × 1.05` → synergistic merge (rare but possible)


In [ ]:
# ── §6: Evaluation harness ───────────────────────────────────────────────

def heuristic_sql_score(sql: str) -> float:
    s = sql.strip().upper()
    if not s: return 0.0
    score = 0.0
    if 'SELECT' in s:     score += 0.25
    if 'FROM' in s:       score += 0.20
    if 10 < len(sql) < 500: score += 0.15
    if '<' not in sql and 'TODO' not in sql.upper(): score += 0.15
    if sql.strip().endswith(';'): score += 0.10
    if 'TRACEBACK' not in sql.upper() and 'ERROR' not in sql.upper(): score += 0.15
    return round(score, 2)


def llm_judge_sql(prompt: str, sql: str, schema: str = '') -> Dict:
    try:
        resp = client.messages.create(
            model=HAIKU, max_tokens=200,
            tools=[{'name': 'submit_score', 'description': 'Submit SQL quality scores',
                    'input_schema': {'type': 'object', 'properties': {
                        'correctness':      {'type': 'integer', 'minimum': 0, 'maximum': 3},
                        'syntax_quality':   {'type': 'integer', 'minimum': 0, 'maximum': 2},
                        'schema_adherence': {'type': 'integer', 'minimum': 0, 'maximum': 2},
                        'efficiency':       {'type': 'integer', 'minimum': 0, 'maximum': 1},
                        'verdict':          {'type': 'string',  'enum': ['correct','partial','incorrect']},
                    }, 'required': ['correctness','syntax_quality','schema_adherence','efficiency','verdict']}}],
            tool_choice={'type': 'tool', 'name': 'submit_score'},
            system='You are a SQL expert. Judge the query strictly.',
            messages=[{'role': 'user', 'content':
                f'Schema: {schema or "standard e-commerce (users, orders, products, order_items)"}\n'
                f'Request: {prompt}\nSQL: {sql}'}])
        sc = resp.content[0].input
        total = sc['correctness'] + sc['syntax_quality'] + sc['schema_adherence'] + sc['efficiency']
        return {'scores': sc, 'total': total, 'pct': total / 8.0}
    except Exception as e:
        return {'scores': {}, 'total': 4, 'pct': 0.5, 'error': str(e)}


# Simulated SQL outputs from 4 model variants
test_cases = [
    {
        'prompt': 'Get all users who placed more than 5 orders, sorted by order count',
        'base':   'SELECT * FROM orders WHERE user_id > 5;',
        'qlora':  'SELECT u.name, COUNT(o.id) AS order_count FROM users u JOIN orders o ON u.id = o.user_id GROUP BY u.id HAVING COUNT(o.id) > 5 ORDER BY order_count DESC;',
        'dpo':    'SELECT u.id, u.name, COUNT(o.id) AS order_count FROM users u INNER JOIN orders o ON u.id = o.user_id GROUP BY u.id, u.name HAVING COUNT(o.id) > 5 ORDER BY order_count DESC;',
        'merged': 'SELECT u.id, u.name, COUNT(o.id) AS order_count FROM users u JOIN orders o ON u.id = o.user_id GROUP BY u.id, u.name HAVING COUNT(o.id) > 5 ORDER BY order_count DESC;',
    },
    {
        'prompt': 'Show top 3 products by revenue this year',
        'base':   'SELECT product FROM products LIMIT 3;',
        'qlora':  'SELECT p.name, SUM(oi.quantity * oi.price) AS revenue FROM products p JOIN order_items oi ON p.id = oi.product_id WHERE YEAR(o.created_at) = YEAR(CURRENT_DATE) GROUP BY p.id ORDER BY revenue DESC LIMIT 3;',
        'dpo':    'SELECT p.id, p.name, SUM(oi.quantity * oi.unit_price) AS total_revenue FROM products p JOIN order_items oi ON p.id = oi.product_id JOIN orders o ON oi.order_id = o.id WHERE YEAR(o.created_at) = YEAR(CURDATE()) GROUP BY p.id, p.name ORDER BY total_revenue DESC LIMIT 3;',
        'merged': 'SELECT p.name, SUM(oi.quantity * oi.unit_price) AS revenue FROM products p JOIN order_items oi ON p.id = oi.product_id JOIN orders o ON oi.order_id = o.id WHERE YEAR(o.created_at) = YEAR(CURDATE()) GROUP BY p.id, p.name ORDER BY revenue DESC LIMIT 3;',
    },
]

print('Evaluating merge quality with heuristic + LLM judge...')
print('=' * 70)
rows = []
for tc in test_cases:
    print(f"\n  Prompt: {tc['prompt'][:55]}...")
    for variant in ['base', 'qlora', 'dpo', 'merged']:
        sql = tc[variant]
        h   = heuristic_sql_score(sql)
        j   = llm_judge_sql(tc['prompt'], sql)
        combined = round(0.4 * h + 0.6 * j['pct'], 3)
        verdict  = j.get('scores', {}).get('verdict', '?')
        print(f'    {variant:>8}: heuristic={h:.2f}  llm={j["pct"]:.2f}  combined={combined:.3f}  [{verdict}]')
        rows.append({'prompt': tc['prompt'][:40], 'model': variant,
                     'heuristic': h, 'llm_pct': j['pct'], 'combined': combined, 'verdict': verdict})

# Summary comparison
df = pd.DataFrame(rows)
pivot = df.pivot_table(index='prompt', columns='model', values='combined', aggfunc='mean')
col_order = [c for c in ['base','qlora','dpo','merged'] if c in pivot.columns]
print('\n=== Combined Score by Model (avg across prompts) ===')
print(pivot[col_order].round(3).to_string())

# Merge quality check
avg = df.groupby('model')['combined'].mean()
if 'merged' in avg.index:
    best_source = max(avg.get('qlora', 0), avg.get('dpo', 0))
    threshold   = 0.9 * best_source
    merged_score = avg['merged']
    status = 'PASS' if merged_score >= threshold else 'FAIL'
    print(f'\nMerge quality gate: merged={merged_score:.3f}  threshold={threshold:.3f}  [{status}]')
# 💡 EXPERIMENT: Replace the SQL strings above with your actual mergekit model outputs


## § 7 — 10 Pitfalls in Model Merging

| # | Pitfall | What goes wrong | Fix |
|---|---|---|---|
| 1 | **Architecture mismatch** | Merge Qwen-0.5B + Llama-3-8B — different shapes | Only merge identical architecture + base |
| 2 | **Different tokenizers** | Vocab IDs don't correspond | Use `--copy-tokenizer` from base |
| 3 | **Blind t=0.5** | Not every capability pair wants equal weight | Sweep t on dev set |
| 4 | **Incompatible tasks** | SQL + code + math = mediocre at all three | Merge related tasks; distill for very different tasks |
| 5 | **Skip DARE before TIES** | Noisy deltas interfere in TIES elect step | Always DARE (p=0.9) before TIES on ≥3 models |
| 6 | **density too low** | Trim 95% → model reverts to base | Use density ≥ 0.2; validate scores |
| 7 | **Chat vs base model merge** | Chat alignment gets diluted | Always merge same tier (both instruct, or both base) |
| 8 | **No post-merge eval** | Passes perplexity, fails task | Run task-specific eval + LLM judge before deploy |
| 9 | **Merge as distillation** | Merge ceiling < distillation quality | Distill for high-stakes tasks; merge for quick wins |
| 10 | **VRAM surprise** | Mergekit loads multiple models simultaneously | Use `--low-cpu-memory --cuda` for 7B+ on T4 |


In [ ]:
# ── §8: Decision tree + Track 3 summary ────────────────────────────────

print('''
╔══════════════════════════════════════════════════════════════════════════╗
║      TRACK 3 DECISION TREE: Choosing Your Technique                     ║
╚══════════════════════════════════════════════════════════════════════════╝

Want consistent OUTPUT FORMAT from the model?
  └─ YES → Fine-tune with QLoRA on format examples (L38)

Want model to PREFER quality outputs over mediocre ones?
  └─ YES → DPO/ORPO preference tuning (L39)

Want a SMALLER, CHEAPER model with big-model quality?
  └─ YES → Knowledge Distillation (L40)
        → White-box: use teacher logits (KL divergence loss)
        → Black-box: train student on teacher completions (SFT)

Have MULTIPLE fine-tuned variants of the same base, want ONE model?
  └─ Similar tasks (SQL variants, code styles)  → Merge L41 SLERP/TIES/DARE ← FREE
  └─ Very different tasks (SQL vs poetry)        → Distill from all teachers

Need to SERVE cheaply at scale?
  └─ YES → vLLM + quantization (L37)
''')

# ── Track 3 summary table ─────────────────────────────────────────────────
summary = pd.DataFrame({
    'Lesson':        ['L37 vLLM', 'L38 QLoRA', 'L39 DPO/ORPO', 'L40 Distillation', 'L41 Merging'],
    'What you get':  [
        '10-24x throughput vs HF generate',
        'Task format / style adaptation',
        'Preference alignment (quality up)',
        'Smaller model = big-model quality',
        'Multiple capabilities, zero training cost',
    ],
    'Training cost': ['None', '~$2-10 (T4)', '~$3-12 (T4)', '~$5-20 (T4)', 'FREE'],
    'Key hyperparams': ['gpu_util, max_model_len', 'r, lora_alpha, lr', 'beta or lambda', 'T, alpha', 'density, t, dare_p'],
})
print(summary.to_string(index=False))

print()
print('🎉 TRACK 3 COMPLETE: vLLM → QLoRA → DPO/ORPO → Distillation → Merging')
print()
print('What is next?')
print()
print('  TRACK 4: Voice + Multimodal Agents')
print('    L42: Realtime API & streaming voice (ASR + LLM + TTS pipeline)')
print('    L43: Image generation as a tool (FLUX / SDXL in agent loop)')
print('    L44: Document AI (PDF extraction, table understanding, Vision+RAG)')
print()
print('  TRACK 5: Agent-ops & Infra')
print('    L42: Durable execution (Temporal / Inngest for long-running agents)')
print('    L43: GPU autoscaling (Modal, RunPod, Replicate)')
print('    L44: OpenTelemetry for LLMs (traces, spans, cost per request)')
print()
print('Reply "Track 4" or "Track 5" to choose. Default: Track 4 on next scheduled run.')


## § 9 — Homework

1. **SLERP t-sweep:** Run `slerp(delta_qlora, delta_dpo, t)` for t in [0.1, 0.3, 0.5, 0.7, 0.9]. Plot recovery score vs t. Where is the sweet spot?

2. **Real mergekit run:** Take your L38 merged model + L39 DPO model. Run `mergekit-yaml` with the DARE-TIES config from this notebook. Evaluate with the SQL harness. Does the merge score beat both individual models?

3. **Three-way merge:** Add your L40 distilled student as a 3rd model in the DARE-TIES config. Does it help or hurt? (Hint: the L40 student was trained on different data — check task compatibility)

4. **Task interference test:** Merge your SQL QLoRA model with a coding model (e.g. `Qwen2.5-Coder-0.5B-Instruct`). Evaluate on both SQL and Python prompts. Does SQL regress? This demonstrates *when not to merge*.

5. **HuggingFace model card:** Push your best model to HuggingFace Hub. Write a `README.md` with: base model, merge method, eval scores, intended use, sample inputs/outputs. This is the start of your **public AI portfolio** — the open-source proof of your expertise.

---
**Track 3 complete!** You can now: run your own LLM cheaply (vLLM), adapt it to your task (QLoRA), align it with preferences (DPO/ORPO), compress it (distillation), and combine capabilities for free (merging). That is the full fine-tuning lifecycle — the same stack used at companies building production LLM systems.
